In [1]:
import pandas as pd
import mysql.connector
from datetime import datetime
import random

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="@statisticsS1",  # Replace with your MySQL password
    database="dental_ai_mvp"
)

# Fetch appointments
df = pd.read_sql("SELECT * FROM appointments", conn)

# Convert date_time to datetime
df['date_time'] = pd.to_datetime(df['date_time'])

# Extract day of week
df['day_of_week'] = df['date_time'].dt.dayofweek  # 0=Monday, 6=Sunday

# One-hot encode appointment_type
df = pd.get_dummies(df, columns=['appointment_type'], drop_first=True)

# Simulate no_show target column
df['no_show'] = [1 if random.random() < 0.2 else 0 for _ in range(len(df))]

# Show first 5 rows
df.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_8260\1890053959.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM appointments", conn)


,appointment_id,patient_id,date_time,past_attendance,booking_lead_time,risk_score,risk_level,day_of_week,appointment_type_Consultation,appointment_type_Extraction,appointment_type_Whitening,no_show
0,1,1,2026-03-03 03:20:20,4,26,0.82,High,1,False,False,True,0
1,2,2,2026-03-21 03:20:20,0,11,NaN,NaN,5,False,True,False,1
2,3,3,2026-02-27 03:20:20,3,16,NaN,NaN,4,False,False,False,0
3,4,4,2026-03-25 03:20:20,0,17,NaN,NaN,2,False,False,True,0
4,5,5,2026-03-27 03:20:20,4,15,NaN,NaN,4,False,True,False,0


In [2]:
# Remove columns we won't use for training
df_model = df.drop(columns=['appointment_id', 'patient_id', 'date_time', 'risk_score', 'risk_level'])

df_model.head()

,past_attendance,booking_lead_time,day_of_week,appointment_type_Consultation,appointment_type_Extraction,appointment_type_Whitening,no_show
0,4,26,1,False,False,True,0
1,0,11,5,False,True,False,1
2,3,16,4,False,False,False,0
3,0,17,2,False,False,True,0
4,4,15,4,False,True,False,0


In [3]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df_model.drop("no_show", axis=1)
y = df_model["no_show"]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

Training data shape: (400, 6)
Test data shape: (100, 6)


In [4]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [5]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.76

Classification Report:

              precision    recall  f1-score   support

           0       0.79      0.94      0.86        77
           1       0.44      0.17      0.25        23

    accuracy                           0.76       100
   macro avg       0.62      0.55      0.55       100
weighted avg       0.71      0.76      0.72       100



In [6]:
import pandas as pd

# Get feature importance
importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

feature_importance_df

,Feature,Importance
1,booking_lead_time,0.486092
2,day_of_week,0.208675
0,past_attendance,0.203610
4,appointment_type_Extraction,0.034992
3,appointment_type_Consultation,0.034572
5,appointment_type_Whitening,0.032060


In [7]:
# Get probability scores
risk_probabilities = model.predict_proba(X_test)[:, 1]

# Create dataframe for review
results_df = X_test.copy()
results_df["Actual_No_Show"] = y_test.values
results_df["Predicted_Probability"] = risk_probabilities

results_df.head()

,past_attendance,booking_lead_time,day_of_week,appointment_type_Consultation,appointment_type_Extraction,appointment_type_Whitening,Actual_No_Show,Predicted_Probability
361,2,14,5,True,False,False,1,0.67
73,5,22,1,True,False,False,0,0.10
374,2,19,5,False,True,False,0,0.26
155,5,16,6,True,False,False,1,0.09
104,0,9,6,False,True,False,1,0.24


In [8]:
import joblib

# Save trained model
joblib.dump(model, "no_show_model.pkl")

print("Model saved successfully.")

Model saved successfully.
